# Quick exploration into `everef` data

Scratch pad for configuring `dlt` pipeline with `everef` data

### Imports

In [1]:
import dlt
from dlt.sources.filesystem import filesystem, read_csv

from dlt.extract import DltResource

from datetime import date

### Sources

In [2]:
BASE_URL = "https://data.everef.net/market-history"
destination = "file://../.local/notebook/data/bronze"
START_DATE = date(2025, 1, 1)


In [3]:
totals_file = f"{BASE_URL}/totals.json"


In [4]:
@dlt.source(name="everef")
def everef_source(year: int, file_glob: str) -> DltResource:
    files = filesystem(
        bucket_url=f"{BASE_URL}/{year}/",
        file_glob=file_glob
    )
    
    market_history = (
        files
        | read_csv(
            chunksize=20_000,
            compression="bz2"
        )
    ).with_name("market_history")

    market_history.apply_hints(
        primary_key=["region_id", "type_id", "date"],
        write_disposition="merge"
    )

    return market_history

In [5]:
pipeline = dlt.pipeline(
    pipeline_name="everef_pipeline",
    # destination="filesystem",
    destination="duckdb",
    dataset_name="everef_history"
)


In [6]:
pipeline.run(
    everef_source(
        2025,
        file_glob="market-history-*.csv.bz2"
    )
)

PipelineStepFailed: Pipeline execution failed at `step=extract` when processing package with `load_id=1777857373.7656927` with exception:

<class 'dlt.extract.exceptions.ResourceExtractionError'>
In processing pipe `filesystem_market_history`: extraction of resource `filesystem_market_history` in `generator` `filesystem` caused an exception: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'

Temp

In [ ]:
import pandas as pd


In [ ]:
df = pd.read_csv("market-history-2026-04-17.csv.bz2")

In [ ]:
df


In [ ]:
df.sort_values(by="volume", ascending=False)

In [ ]:
df[df.region_id == 10000002]
